# Onechess : fin du réseau final sur TPU (Colab)

Reprend le réseau final là où la session GPU de Kaggle s'est arrêtée, et le finit sur le TPU gratuit de Colab (~3 fois une T4).

**À faire :** *Exécution → Modifier le type d'exécution → TPU v5e-1*, puis *Exécution → Tout exécuter*, et laisser l'onglet ouvert. Colab demande l'accès à Google Drive : la progression y est sauvée (`MyDrive/onechess_final/`). Si Colab se déconnecte, *Tout exécuter* reprend à la dernière époque finie. Aucune clé ni mot de passe.

In [ ]:
# 1. Drive et code (ne pas importer jax ici : il reserverait le TPU)
import os, sys, glob, io, tarfile, urllib.request, subprocess, time
from google.colab import drive
drive.mount('/content/drive')
SORTIE = '/content/drive/MyDrive/onechess_final'
os.makedirs(SORTIE, exist_ok=True)
subprocess.run('rm -rf /content/code && mkdir -p /content/code', shell=True)
tarfile.open(fileobj=io.BytesIO(urllib.request.urlopen('https://codeload.github.com/ovnice-dev/onechess/tar.gz/main').read())).extractall('/content/code')
TOOLS = glob.glob('/content/code/*/tools')[0]
try:
    import optax
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optax'], check=True)
print('sauvegarde :', SORTIE, os.listdir(SORTIE))

In [ ]:
# 2. Donnees : exactement celles du final sur Kaggle (7 premiers Go, memes graines, 4 parts, meme validation)
t0 = time.time()
if not os.path.exists('/content/donnees/z_val.bin'):
    subprocess.run([sys.executable, '-u', TOOLS + '/hf_sources.py', '--url',
                    'https://huggingface.co/datasets/official-stockfish/master-binpacks/resolve/main/nodes5000pv2_UHO.binpack',
                    '--prefixe', 'n5', '--dest', '/content/donnees', '--octets', '7000000000', '--garde', '1.0',
                    '--fixe', '240', '--graine', '11', '--morceau', '4000000000', '--parts', '4'], check=True)
    import numpy as np
    v = sorted(glob.glob('/content/donnees/*_p00_*.bin'))[0]
    os.rename(v, '/content/donnees/z_val.bin')
    np.fromfile('/content/donnees/z_val.bin', dtype='uint8').reshape(-1, 32)[:5_000_000].tofile('/content/donnees/z_val.bin')
n = sum(os.path.getsize(f) for f in glob.glob('/content/donnees/*.bin')) // 32
print(f'{n:,} positions, {len(glob.glob("/content/donnees/*.bin"))} fichiers, {time.time()-t0:.0f} s')

In [ ]:
# 3. Point de reprise de la session GPU de Kaggle (une seule fois ; ensuite on reprend depuis Drive)
REPRISE = '/content/final_gpu.ckpt'
if not os.path.exists(SORTIE + '/final_tpu.pkl') and not os.path.exists(REPRISE):
    urllib.request.urlretrieve('https://github.com/ovnice-dev/onechess/releases/download/final-reprise/final.ckpt', REPRISE)
    print('point de reprise GPU telecharge :', os.path.getsize(REPRISE) // 1_000_000, 'Mo')

In [ ]:
# 4. Entrainement sur TPU (affichage en direct, journal sur Drive). Relancer cette cellule reprend.
!python {TOOLS}/train_jax.py --ft-dense 1 --stream "/content/donnees/*.bin" --hidden 1024 --buckets mat4 --out-buckets 4 --epochs 60 --epoch-files 60 --chunk-files 10 --reprise-torch {REPRISE} --checkpoint {SORTIE}/final_tpu.pkl --ckpt-torch {SORTIE}/final_tpu.ckpt --out {SORTIE}/final.nnue --save-pt {SORTIE}/final.pt 2>&1 | tee -a {SORTIE}/final_tpu.log

Quand la ligne `reseau ecrit` apparaît, le réseau final est dans `MyDrive/onechess_final/final.nnue`.